<a href="https://colab.research.google.com/github/pcmouadji-dot/deep_learning/blob/main/comment_toxic.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [35]:
import os
import pandas as pd
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.layers import TextVectorization
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dropout, Bidirectional, Dense, Embedding

In [36]:
df=pd.read_csv('train.csv', engine='python', on_bad_lines='skip')
df.head()
#df[df['toxic']==1].head()
test=pd.read_csv('test.csv', engine='python', on_bad_lines='skip')

In [37]:
x=df['comment_text']
y=df[df.columns[2:]].values



In [38]:
x.values.shape

(159571,)

In [39]:
vecto=TextVectorization(max_tokens=250000,output_mode='int', output_sequence_length=1800)
vecto.adapt(x.values)


In [40]:
vecto('bitch')

<tf.Tensor: shape=(1800,), dtype=int64, numpy=array([762,   0,   0, ...,   0,   0,   0])>

In [41]:
vec_text=vecto(x.values)
vec_text

<tf.Tensor: shape=(159571, 1800), dtype=int64, numpy=
array([[   645,     76,      2, ...,      0,      0,      0],
       [219427,     54,   2489, ...,      0,      0,      0],
       [   425,    441,     70, ...,      0,      0,      0],
       ...,
       [ 32445,   7392,    383, ...,      0,      0,      0],
       [     5,     12,    534, ...,      0,      0,      0],
       [     5,      8,    130, ...,      0,      0,      0]])>

In [42]:
dataset=tf.data.Dataset.from_tensor_slices((vec_text,y))
dataset=dataset.cache()
dataset=dataset.shuffle(90000)
dataset=dataset.batch(9)
dataset=dataset.prefetch(5)

In [43]:
train=dataset.take(int(len(dataset)*.8))
val=dataset.skip(int(len(dataset)*.8)).take(int(len(dataset)*.2))
#test=dataset.skip(int(len(dataset)*.9)).take(int(len(dataset)*.1))

In [44]:
train_generater=train.as_numpy_iterator()
train_generater.next()

(array([[132014,   6387,     36, ...,      0,      0,      0],
        [   113,      4,     42, ...,      0,      0,      0],
        [   235,    281,    376, ...,      0,      0,      0],
        ...,
        [    31,      2,    145, ...,      0,      0,      0],
        [     8,     69,      2, ...,      0,      0,      0],
        [   265,     35,   2891, ...,      0,      0,      0]]),
 array([[0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0]]))

In [45]:
model=Sequential([
    Embedding(250000+1,32),
    Bidirectional(LSTM(32,activation='tanh')),
    Dense(128,activation='relu'),
    Dense(256,activation='relu'),
    Dense(128,activation='relu'),
    Dense(6,activation='sigmoid')


])
model.compile(loss='BinaryCrossentropy',optimizer='Adam')

model.summary()#before trainning

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_2 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_2 (Bidirectional) │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_10 (Dense)                │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_11 (Dense)                │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [46]:
history=model.fit(train,epochs=2,validation_data=val)
model.summary()

Epoch 1/2
14184/14184 ━━━━━━━━━━━━━━━━━━━━ 1619s 114ms/step - loss: 0.0601 - val_loss: 0.0457
Epoch 2/2
14184/14184 ━━━━━━━━━━━━━━━━━━━━ 1557s 110ms/step - loss: 0.0442 - val_loss: 0.0370


Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_2 (Embedding)         │ (None, 1800, 32)       │     8,000,032 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_2 (Bidirectional) │ (None, 64)             │        16,640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 128)            │         8,320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, 256)            │        33,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_10 (Dense)                │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_11 (Dense)                │ (None, 6)              │           774 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 24,275,060 (92.60 MB)

 Trainable params: 8,091,686 (30.87 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 16,183,374 (61.73 MB)

In [47]:
txt=vecto('u are gay')
res=model.predict(np.expand_dims(txt,0))
res

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 340ms/step


array([[0.86576194, 0.01713142, 0.21052015, 0.04701968, 0.33919728,
        0.07665876]], dtype=float32)

In [ ]:
#id=test['id']
#test=test.drop(columns=['id'])


In [ ]:
#test_text = vecto(test['comment_text'])
#test_dataset = tf.data.Dataset.from_tensor_slices(test_text)
#test_dataset = test_dataset.batch(9)


In [ ]:
#test_predictions = model.predict(test_dataset)
#display(test_predictions[:5]) # Display first 5 predictions

In [50]:
!pip install gradio jinja2
import gradio as gr



In [51]:
model.save('tanamor.h5')


In [52]:
def score_comment(comment):
    vectorized_comment = vecto(np.expand_dims(comment, 0))
    results = model.predict(vectorized_comment)
    result=''
    for ind ,col in enumerate(df.columns[2:]):
        result+='{}: {}\n'.format(col,results[0][ind]>0.5)
    return result


In [54]:
face=gr.Interface(fn=score_comment,inputs=gr.Textbox(lines=2,placeholder='comment here'),outputs='text')
face.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://54bd2bc9218c6820bf.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
